# PRACTICE 2: PREDICTING PRODUCT SALES
## CHẶNG 1: KHÁM PHÁ DỮ LIỆU & TIỀN XỬ LÝ (DATA CLEANING & EXPLORATORY DATA ANALYSIS)

> **Mục tiêu kỹ thuật:**
> - Khảo sát cấu trúc 128.975 giao dịch thương mại điện tử Amazon Ấn Độ (`Amazon Sale Report.csv`).
> - Trả lời 5 câu hỏi vàng EDA: Cấu trúc, Khuyết thiếu, Phân phối đơn biến Amount, Phân tích đa biến địa lý/mùa vụ, và Quét ngoại lai IQR.
> - Làm sạch dữ liệu, loại bỏ đơn hủy/lỗi, khử rò rỉ dữ liệu và xuất 102.852 dòng sạch (`clean_amazon_sales.csv`).

### 1. Khảo sát Cấu trúc Tổng quan & Đơn biến Biến mục tiêu (Amount) — Nguyễn Xuân Đông (Trưởng nhóm - 16%)
> **Trạng thái:** Đã hoàn thiện sẵn sàng.
- Khảo sát shape, kiểu dữ liệu, tỷ lệ khuyết thiếu và số dòng trùng lặp qua hàm `inspect_data_structure`.
- Đối chiếu phân phối lệch phải của Amount trước và sau biến đổi Log `np.log1p` qua hàm `get_univariate_stats`.

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', palette='muted')

DATA_RAW_PATH = Path('../data/raw/Amazon Sale Report.csv')
df_raw = pd.read_csv(DATA_RAW_PATH, low_memory=False)
print(f"✓ Kích thước dữ liệu thô: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")
display(df_raw.head(3))

In [ ]:
# TODO: [Mai An Thịnh - 14%] Phân tích đa biến theo Ngành hàng, Bang và Khách sỉ B2B
# Triển khai hàm analyze_bivariate_sales trong data/scripts/eda.py để hoàn thiện phân tích
from data.scripts.eda import analyze_bivariate_sales

biv_stats = analyze_bivariate_sales(df_raw)
print("✓ Phân tích hai biến thành công:", list(biv_stats.keys()))

# Trực quan hóa doanh số trung vị theo Ngành hàng và Top 8 Bang lớn nhất
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_sales = df_raw.groupby('Category')['Amount'].median().sort_values(ascending=False)
sns.barplot(x=cat_sales.values, y=cat_sales.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Doanh số trung vị theo Ngành hàng', fontweight='bold')

top_states = df_raw['ship-state'].value_counts().head(8)
sns.barplot(x=top_states.values, y=top_states.index, ax=axes[1], palette='crest')
axes[1].set_title('Top 8 Bang có đơn đặt hàng lớn nhất', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: [Mai An Thịnh - 14%] Quét ngoại lai doanh số Amount theo công thức IQR tiêu chuẩn
# Triển khai hàm detect_sales_outliers_iqr trong data/scripts/eda.py để phát hiện ranh giới ngoại lai
from data.scripts.eda import detect_sales_outliers_iqr

iqr_res = detect_sales_outliers_iqr(valid_amounts)
print(f"• Ngưỡng IQR: Q1={iqr_res['q1']:.1f}, Q3={iqr_res['q3']:.1f}, IQR={iqr_res['iqr']:.1f}")
print(f"• Khoảng hợp lệ không ngoại lai: [{iqr_res['lower_bound']:.1f}, {iqr_res['upper_bound']:.1f}] INR")
print(f"• Số lượng mẫu ngoại lai: {iqr_res['n_outliers']:,} ({iqr_res['outlier_pct']:.2f}%)")

### 2. Khám phá Chuyên sâu — Đa biến, Địa lý, Mùa vụ & Ngoại lai IQR — Mai An Thịnh (14%)
> **Nhiệm vụ Mai An Thịnh (14%):** Hoàn thiện các hàm `detect_iqr_outliers` và `get_bivariate_analysis` trong `data/scripts/eda.py`.
- Khảo sát doanh số theo Ngành hàng (Category), Kênh hoàn tất (Fulfilment) và Top 10 bang.
- Tính toán ranh giới ngoại lai IQR trên biến Amount.

In [ ]:
# TODO: [Mai An Thịnh - 14%] Phân tích đa biến theo Ngành hàng và Kênh hoàn tất/Địa phương
# Gợi ý: Sử dụng hàm get_bivariate_analysis(df_raw) từ data/scripts/eda.py hoặc vẽ qua Seaborn
# Phân bố đơn biến và đa biến theo Ngành hàng và Kích cỡ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_sales = df_raw.groupby('Category')['Amount'].median().sort_values(ascending=False)
sns.barplot(x=cat_sales.values, y=cat_sales.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Doanh số trung vị theo Ngành hàng', fontweight='bold')

top_states = df_raw['ship-state'].value_counts().head(8)
sns.barplot(x=top_states.values, y=top_states.index, ax=axes[1], palette='crest')
axes[1].set_title('Top 8 Bang có đơn đặt hàng lớn nhất', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: [Mai An Thịnh - 14%] Quét ngoại lai doanh số Amount theo công thức IQR tiêu chuẩn
# Gợi ý: Sử dụng hàm detect_iqr_outliers(valid_amounts) từ data/scripts/eda.py
# Quét ngoại lai theo công thức IQR tiêu chuẩn
q1 = valid_amounts.quantile(0.25)
q3 = valid_amounts.quantile(0.75)
iqr = q3 - q1
lower_bound = max(0, q1 - 1.5 * iqr)
upper_bound = q3 + 1.5 * iqr
outliers = valid_amounts[(valid_amounts < lower_bound) | (valid_amounts > upper_bound)]

print(f"• Ngưỡng IQR: Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}")
print(f"• Khoảng hợp lệ không ngoại lai: [{lower_bound:.1f}, {upper_bound:.1f}] INR")
print(f"• Số lượng mẫu ngoại lai: {len(outliers):,} ({len(outliers)/len(valid_amounts)*100:.2f}%)")

In [ ]:
# Bảng đúc kết kết luận EDA 5 câu hỏi định hướng cho Tiền xử lý
eda_conclusions = pd.DataFrame([
    {"Vấn đề EDA": "Đơn hàng hủy (Status=Cancelled)", "Tác động": "Amount bị null hoặc không sinh doanh thu thực", "Hành động": "Lọc bỏ toàn bộ đơn hủy"},
    {"Vấn đề EDA": "Amount <= 0 hoặc Qty <= 0", "Tác động": "Nhiễu dữ liệu quà tặng/đổi trả", "Hành động": "Chỉ giữ đơn Amount > 0 và Qty > 0"},
    {"Vấn đề EDA": "Cột định danh & Zero-variance", "Tác động": "Gây quá khớp hoặc vô nghĩa", "Hành động": "Loại bỏ Order ID, ASIN, currency, ship-country"},
    {"Vấn đề EDA": "Amount lệch phải dài (Skewness > 1.5)", "Tác động": "Mô hình tuyến tính học thiên lệch", "Hành động": "Huấn luyện trên log1p(Amount) và nghịch đảo expm1 khi kiểm thử"},
    {"Vấn đề EDA": "ship-state có > 50 bang phân mảnh", "Tác động": "Bùng nổ số chiều khi OneHot", "Hành động": "Gom nhóm Top 10 bang lớn + 'Other'"}
])
display(eda_conclusions)

### 3. Tiền xử lý Dữ liệu & Xuất tệp Sạch (Data Cleaning) — Nguyễn Minh Nhật (14%)
> **Nhiệm vụ Nguyễn Minh Nhật (14%):** Hoàn thiện module `data/scripts/data_preprocessing.py` (`clean_amazon_sales_data`, `handle_missing_and_duplicates`).
- Lọc bỏ đơn hàng hủy và đơn có Amount <= 0, Qty <= 0.
- Điền khuyết thiếu cho `promotion-ids`, `Courier Status`, `ship-state`.
- Khử trùng lặp triệt để và xuất `clean_amazon_sales.csv` (102.852 dòng sạch).

In [ ]:
# TODO: [Nguyễn Minh Nhật - 14%] Gọi hàm clean_amazon_sales_data để làm sạch và xuất 102.852 dòng sạch ra data/processed/
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from data.scripts.data_preprocessing import clean_amazon_sales_data

RAW_CSV = Path('../data/raw/Amazon Sale Report.csv')
CLEAN_CSV = Path('../data/processed/clean_amazon_sales.csv')

df_clean = clean_amazon_sales_data(RAW_CSV, CLEAN_CSV)
print(f"✓ Đã làm sạch thành công: {len(df_clean):,} dòng dữ liệu sạch được lưu tại '{CLEAN_CSV.name}'.")
assert len(df_clean) == 102852, f"Số dòng không khớp chuẩn: {len(df_clean)} != 102,852"
assert df_clean.isnull().sum().sum() == 0, "Vẫn còn giá trị NaN trong tập dữ liệu sạch!"
display(df_clean.head(3))